In [5]:
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [6]:
import os

print(os.getenv("OPENAI_API_KEY") is not None)

True


In [8]:
from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [9]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [10]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

In [11]:
chain = prompt | llm

In [12]:
store = {}

In [13]:
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [15]:
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

/Users/xianqiu/Desktop/openai-demo/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
response = conversation.invoke(
    {"input": "Hi, my name is Andrew"},
    config={"configurable": {"session_id": "user_1"}}
)

print(response.content)

Hi Andrew! How can I assist you today?


In [17]:
response = conversation.invoke(
    {"input": "What is 1+1?"},
    config={"configurable": {"session_id": "user_1"}}
)

print(response.content)

1 + 1 equals 2.


In [18]:
response = conversation.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user_1"}}
)

print(response.content)

Your name is Andrew.


In [19]:
store["user_1"].messages

[HumanMessage(content='Hi, my name is Andrew', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi Andrew! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 23, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2ef023509e', 'id': 'chatcmpl-DhTG4XZeH7HSk6nuYvUoNKdfM74xD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e43bd-5b0f-7233-91ad-aa8c0a0c9655-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 23, 'output_tokens': 10, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),

In [20]:
for message in store["user_1"].messages:
    print(type(message).__name__, ":", message.content)

HumanMessage : Hi, my name is Andrew
AIMessage : Hi Andrew! How can I assist you today?
HumanMessage : What is 1+1?
AIMessage : 1 + 1 equals 2.
HumanMessage : What is my name?
AIMessage : Your name is Andrew.
